# W09 — Assignment único semanal (Limpieza avanzada + Quality Gates)

## Setup

In [1]:
from pathlib import Path
import duckdb

PROJECT_ROOT = Path(".").resolve()
DB_PATH = PROJECT_ROOT / "data" / "exoplanets.duckdb"
RAW_CSV = PROJECT_ROOT / "data" / "raw" / "pscomppars.csv"

if not DB_PATH.exists():
    raise FileNotFoundError(f"Missing {DB_PATH}. Run W06 pipeline first.")

con = duckdb.connect(str(DB_PATH))

def sql_path(p: Path) -> str:
    return "'" + p.resolve().as_posix().replace("'","''") + "'"

if not RAW_CSV.exists():
    raise FileNotFoundError(f"Missing {RAW_CSV}")

con.execute("DROP VIEW IF EXISTS raw_ps")
con.execute(f"CREATE VIEW raw_ps AS SELECT * FROM read_csv_auto({sql_path(RAW_CSV)})")

## Parte A — Limpieza avanzada

In [2]:
# TODO 1: method_synonyms(raw_norm, canonical)
# - raw_norm: normalizado (LOWER/TRIM + colapsar espacios si quieres)
# - canonical: snake_case
# - >= 6 filas

con.execute("DROP TABLE IF EXISTS method_synonyms")
con.execute("CREATE TABLE method_synonyms(raw_norm VARCHAR, canonical VARCHAR)")
con.execute("""
INSERT INTO method_synonyms VALUES
  ('transit','transit'), ('radial velocity','radial_velocity'),
  ('imaging','imaging'), ('microlensing','microlensing'),
  ('timing','timing'), ('astrometry','astrometry')
""")
con.sql("SELECT * FROM method_synonyms").show()

┌─────────────────┬─────────────────┐
│    raw_norm     │    canonical    │
│     varchar     │     varchar     │
├─────────────────┼─────────────────┤
│ transit         │ transit         │
│ radial velocity │ radial_velocity │
│ imaging         │ imaging         │
│ microlensing    │ microlensing    │
│ timing          │ timing          │
│ astrometry      │ astrometry      │
└─────────────────┴─────────────────┘



In [3]:
# TODO 2: silver_planet_v3
# Debe incluir:
# - hostname_canon
# - discoverymethod_canon (synonyms + COALESCE fallback)
# - disc_year_int = TRY_CAST(disc_year AS INTEGER)
# - disc_year_bad flag

con.execute("DROP TABLE IF EXISTS silver_planet_v3")
con.execute("""
CREATE TABLE silver_planet_v3 AS
SELECT
  LOWER(TRIM(hostname)) AS hostname_canon,
  COALESCE(s.canonical, LOWER(TRIM(discoverymethod))) AS discoverymethod_canon,
  TRY_CAST(disc_year AS INTEGER) AS disc_year_int,
  (TRY_CAST(disc_year AS INTEGER) IS NULL AND disc_year IS NOT NULL) AS disc_year_bad
FROM raw_ps
LEFT JOIN method_synonyms s ON LOWER(TRIM(discoverymethod)) = s.raw_norm
WHERE pl_name IS NOT NULL
""")
print("Filas en silver_planet_v3:", con.execute("SELECT COUNT(*) FROM silver_planet_v3").fetchone()[0])
bad = con.execute("SELECT COUNT(*) FROM silver_planet_v3 WHERE disc_year_bad").fetchone()[0]
print(f"Años no convertibles: {bad}")

Filas en silver_planet_v3: 6291
Años no convertibles: 0


## Parte B — Quality gates

In [5]:
from datetime import datetime, timezone
from pathlib import Path

In [6]:
# TODO 3: quality_events + 4 checks
# Crea tabla:
# ts_utc, check_name, status, metric_value, details

con.execute("DROP TABLE IF EXISTS quality_events")
con.execute("""
CREATE TABLE quality_events(
  ts_utc TIMESTAMP,
  check_name VARCHAR,
  status VARCHAR,
  metric_value DOUBLE,
  details VARCHAR
)
""")

ts = datetime.now(timezone.utc).isoformat()
# Check 1: unicidad de pl_name (desde raw_ps)
dup_count = con.execute("SELECT COUNT(*) - COUNT(DISTINCT pl_name) FROM raw_ps WHERE pl_name IS NOT NULL").fetchone()[0]
status = "PASS" if dup_count == 0 else "FAIL"
con.execute("INSERT INTO quality_events VALUES (?, 'uniqueness_pl_name', ?, ?, 'duplicates')", [ts, status, dup_count])

# Check 2: nulos en hostname
null_host = con.execute("SELECT COUNT(*) - COUNT(hostname) FROM raw_ps").fetchone()[0]
status = "PASS" if null_host == 0 else "FAIL"
con.execute("INSERT INTO quality_events VALUES (?, 'nulls_hostname', ?, ?, 'null hostname')", [ts, status, null_host])

# Check 3: años válidos
invalid_years = con.execute("SELECT COUNT(*) FROM raw_ps WHERE disc_year < 1980 OR disc_year > 2026").fetchone()[0]
status = "WARN" if invalid_years > 0 else "PASS"
con.execute("INSERT INTO quality_events VALUES (?, 'valid_disc_year', ?, ?, 'years out of range')", [ts, status, invalid_years])

# Check 4: pl_rade positivo
neg_rade = con.execute("SELECT COUNT(*) FROM raw_ps WHERE pl_rade <= 0").fetchone()[0]
status = "PASS" if neg_rade == 0 else "FAIL"
con.execute("INSERT INTO quality_events VALUES (?, 'positive_pl_rade', ?, ?, 'non-positive radii')", [ts, status, neg_rade])

con.sql("SELECT * FROM quality_events").show()

# Exportar a CSV
ART_DIR = PROJECT_ROOT / "artifacts"
ART_DIR.mkdir(exist_ok=True)
out = ART_DIR / "w09_quality_events.csv"
con.execute(f"COPY quality_events TO '{out}' (HEADER, DELIMITER ',')")
print(f"Exportado a {out}")

con.close()

┌────────────────────────────┬────────────────────┬─────────┬──────────────┬────────────────────┐
│           ts_utc           │     check_name     │ status  │ metric_value │      details       │
│         timestamp          │      varchar       │ varchar │    double    │      varchar       │
├────────────────────────────┼────────────────────┼─────────┼──────────────┼────────────────────┤
│ 2026-05-28 21:46:58.142515 │ uniqueness_pl_name │ PASS    │          0.0 │ duplicates         │
│ 2026-05-28 21:46:58.142515 │ nulls_hostname     │ PASS    │          0.0 │ null hostname      │
│ 2026-05-28 21:46:58.142515 │ valid_disc_year    │ PASS    │          0.0 │ years out of range │
│ 2026-05-28 21:46:58.142515 │ positive_pl_rade   │ PASS    │          0.0 │ non-positive radii │
└────────────────────────────┴────────────────────┴─────────┴──────────────┴────────────────────┘

Exportado a C:\Users\Ider Diaz\Desktop\Todos los Notebooks\artifacts\w09_quality_events.csv


## Entregable único semanal (W09)

Entrega:
- `assignments/W09_assignment_student.ipynb` ejecutado
- `docs/w09_report.md` (usar template)
- `docs/w09_quality.md` (usar template)
- 1 entrada en `docs/decisions_log.md` (usar template)